---
# FASE 3 | PREPARACION DE LOS DATOS (Data Preparation)
---

## 3.1 Limpieza de datos

In [ ]:
df = df_raw.copy()

# Eliminar duplicados
antes = len(df)
df = df.drop_duplicates()
print(f'Duplicados eliminados: {antes - len(df)}')

# Eliminar registros con views = 0 (no aportan informacion de engagement)
df = df[df['views'] > 0].reset_index(drop=True)
print(f'Registros con views=0 eliminados: {antes - len(df)}')
print(f'Dataset limpio: {len(df):,} registros')

## 3.2 Construcción de la variable objetivo: VIRALIDAD

### Metodología:

La variable `viralidad` se construye en 4 pasos:

1. **Transformacion logaritmica** (`log1p`) de views, likes, comments, shares para reducir el sesgo de la distribucion de cola larga.
2. **Rango percentil** de cada metrica transformada: ubica cada video en una posicion relativa (0 = menor, 1 = mayor) dentro del conjunto de datos.
3. **Puntaje compuesto ponderado**:

$$\text{engagement\_score} = 0.30 \cdot r_{views} + 0.25 \cdot r_{likes} + 0.20 \cdot r_{comments} + 0.25 \cdot r_{shares}$$

   Los pesos reflejan que las vistas (alcance) y los shares (propagacion) son los principales indicadores de viralidad, seguidos de likes y comentarios.

4. **Clasificacion binaria**: videos con `engagement_score >= percentil 50` son etiquetados como **virales** (1); los demas como **no virales** (0).

In [ ]:
df = build_viralidad(df, threshold_pct=50)

In [ ]:
# Visualizar el puntaje de engagement y el corte de viralidad
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Distribucion del puntaje
umbral = df[df['viralidad'] == 1]['engagement_score'].min()
axes[0].hist(df['engagement_score'], bins=60, color='#4C72B0', alpha=0.7, edgecolor='white')
axes[0].axvline(umbral, color='red', linestyle='--', lw=2, label=f'Umbral viral = {umbral:.3f}')
axes[0].set_title('Distribucion del Puntaje de Engagement', fontweight='bold')
axes[0].set_xlabel('engagement_score')
axes[0].set_ylabel('Frecuencia')
axes[0].legend()

# Balance de clases
class_counts = df['viralidad'].value_counts()
labels = ['No Viral (0)', 'Viral (1)']
colors = ['#C44E52', '#55A868']
bars = axes[1].bar(labels, class_counts.values, color=colors, alpha=0.85)
for bar, val in zip(bars, class_counts.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
                 f'{val:,}\n({val/len(df)*100:.1f}%)',
                 ha='center', va='bottom', fontsize=10)
axes[1].set_title('Balance de Clases - Variable Viralidad', fontweight='bold')
axes[1].set_ylabel('Cantidad de Videos')

plt.suptitle('Variable Objetivo: Viralidad', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/viralidad_distribucion.png', bbox_inches='tight')
plt.show()

## 3.3 Ingeniería de características

**Criterio de diseño:** Para evitar fuga de datos (*data leakage*), los modelos NO usan los valores brutos de views, likes, comments y shares como caracteristicas (ya que estos fueron usados para construir `viralidad` directamente). En cambio, se generan **tasas de engagement relativo** que miden la *eficiencia* de interaccion, que es conceptualmente distinta del alcance absoluto:

$$\text{likes\_per\_view} = \frac{\text{likes}}{\text{views}}$$

Un video puede tener pocos views pero una tasa de likes muy alta (contenido de nicho muy comprometido) o muchos views con una tasa de likes baja (contenido viral en alcance pero no en engagement cualitativo). Esta distincion hace el problema genuinamente interesante desde el punto de vista predictivo.

In [ ]:
df = build_features(df)

feature_info = get_feature_columns()
print('Caracteristicas por tipo:')
for tipo, cols in feature_info.items():
    print(f'  {tipo:12}: {cols}')

In [ ]:
# Visualizar tasas de engagement por clase
rate_cols = ['likes_per_view', 'comments_per_view', 'shares_per_view', 'engagement_rate']
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, col in zip(axes, rate_cols):
    for label, color in [(0, '#C44E52'), (1, '#55A868')]:
        subset = df[df['viralidad'] == label][col]
        ax.hist(np.log1p(subset), bins=40, alpha=0.6, color=color,
                label='No Viral' if label == 0 else 'Viral', density=True)
    ax.set_title(f'log1p({col})', fontsize=9, fontweight='bold')
    ax.set_xlabel('Valor')
    ax.legend(fontsize=8)

plt.suptitle('Distribucion de Tasas de Engagement por Clase (Viral vs No Viral)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/engagement_rates_por_clase.png', bbox_inches='tight')
plt.show()

## 3.4 Partición train / test y preprocesamiento

In [ ]:
feature_info = get_feature_columns()
cat_cols = feature_info['categorical']
num_cols = feature_info['numerical']
bin_cols = feature_info['binary']
all_features = cat_cols + num_cols + bin_cols

X_train, X_test, y_train, y_test = split_data(
    df, feature_cols=all_features, target_col='viralidad',
    test_size=0.20, random_state=RANDOM_STATE
)

In [ ]:
# Construir y ajustar el preprocesador sobre los datos de entrenamiento
preprocessor = build_preprocessor(cat_cols, num_cols, bin_cols)
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc  = preprocessor.transform(X_test)

save_preprocessor(preprocessor, path='../models/preprocessor.joblib')

# Nombres de caracteristicas tras el preprocesamiento
ohe_names = preprocessor.named_transformers_['cat'].get_feature_names_out(cat_cols).tolist()
feature_names_proc = ohe_names + num_cols + bin_cols

print(f'\nDimensiones post-preprocesamiento:')
print(f'  X_train: {X_train_proc.shape}')
print(f'  X_test : {X_test_proc.shape}')
print(f'  Total features: {len(feature_names_proc)}')

In [ ]:
# Guardar datos procesados para reproducibilidad
cols_to_save = all_features + ['viralidad', 'engagement_score']
df[cols_to_save].to_csv('../data/processed/data_processed.csv', index=False)
print('Datos procesados guardados en data/processed/data_processed.csv')